In [ ]:
# =============================================================================
# OULAD CROSS-CONTEXT REPLICATION  -  Sections 4.16 and 5.12, Table 4, Table S9
#
# This notebook is self-contained: it produces every OULAD figure reported in
# the manuscript and ASSERTS each one. If a number drifts, the cell stops.
#
# WHAT IT WRITES (commit all four):
#   outputs/oulad_per_seed_metrics.csv      multi-seed AUC, Section 5.12
#   outputs/oulad_seed42_predictions.csv    seed-42 probabilities + imd_band
#   outputs/run_log_oulad.json              thresholds, gaps, decision record
#   outputs/oulad_shap_values.csv           SHAP ranking behind the pruning
#
# TWO CORRECTIONS over earlier versions of this notebook, both of which changed
# reported numbers:
#   1. The screening threshold is selected on a 25% VALIDATION split of the
#      training partition (Section 4.4). An earlier version selected it on the
#      held-out test set, giving tau = 0.4416 / 0.4533 instead of 0.4719 /
#      0.4874 and gaps of 0.121 / 0.148 instead of 0.119 / 0.160.
#   2. Band eligibility is SUBGROUP SIZE, at least 30 students in the
#      evaluation partition (Section 4.9), and the Missing deprivation category
#      is excluded as a missing-data category rather than a deprivation level.
#      An earlier version gated on the at-risk count, which Section 4.9
#      explicitly disclaims.
#
# DATA: SOURCE_DATA is the institutional extract. It is NOT in this repository
# and must never be committed (see .gitignore). CELL 7 reads it only for the
# population-shift diagnostics. OULAD itself is public: Kuzilek, Hlosta and
# Zdrahal (2017), CC BY 4.0. kaggle.json carries an API token - never commit it.
# =============================================================================
!pip install shap imbalanced-learn pyarrow -q

import os, re, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, precision_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import xgboost as xgb
from scipy.stats import chi2, rankdata
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

SOURCE_DATA = "/content/drive/MyDrive/model_ready_dataset_1.xlsx"   # never committed
OUT_DIR     = "/content/drive/MyDrive/cinque_oulad"
REPO_OUT    = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPO_OUT, exist_ok=True)

SEEDS         = [42, 0, 123, 2024, 7]      # Section 4.8
TARGET_RECALL = 0.75                       # Section 4.4
MIN_BAND_N    = 30                         # Section 4.9, subgroup SIZE
MISSING_BAND  = "Missing"                  # excluded, Section 4.9

_fail = []
def check(label, got, want, tol=5e-4, fmt="{:.4f}"):
    """Print GOT against the published value and record any mismatch."""
    ok = abs(float(got) - float(want)) <= tol
    print(f"  {label:<46s} {fmt.format(got):>10}   paper {fmt.format(want):>10}   "
          f"{'MATCH' if ok else 'DIFFERS'}")
    if not ok:
        _fail.append(f"{label}: got {got}, paper {want}")
    return ok

def seal(cell_name):
    """Stop the notebook if anything in this cell drifted."""
    if _fail:
        msgs = list(_fail); _fail.clear()
        raise AssertionError(f"{cell_name} does not reproduce the paper:\n  " + "\n  ".join(msgs))
    print(f"  [{cell_name}] all checks passed")

print("setup ok. Drive outputs ->", OUT_DIR, "| repo outputs ->", REPO_OUT)

In [ ]:
# CELL 1 - fetch OULAD from Kaggle (public, CC BY 4.0)
# kaggle.json holds an API token. Keep it out of the repository.
!pip install kaggle -q
from google.colab import files
print("Upload kaggle.json  (Kaggle -> Settings -> API -> Create New Token)")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle*.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d anlgrbz/student-demographics-online-education-dataoulad -p /content --unzip
!mkdir -p /content/oulad && mv /content/*.csv /content/oulad/ 2>/dev/null
!ls /content/oulad

In [ ]:
# CELL 2 - label. Withdrawn only, to match the source construct of at-risk.
info = pd.read_csv("/content/oulad/studentInfo.csv")
info["target"] = (info.final_result == "Withdrawn").astype(int)
print("studentInfo:", info.shape)
print(info.final_result.value_counts(normalize=True).round(3).to_string())
print(f"\nOULAD raw: n={len(info):,}, at-risk={info.target.sum():,} ({info.target.mean()*100:.1f}%)")
print( "Source:    n=3,325, at-risk=565 (17.0%)   [for comparison]")

In [ ]:
# CELL 3 - assessment features (TMA only), comparable to the source module scores
sa   = pd.read_csv("/content/oulad/studentAssessment.csv")
asmt = pd.read_csv("/content/oulad/assessments.csv")
sa = sa.merge(asmt[["id_assessment","code_module","code_presentation","assessment_type","date"]],
              on="id_assessment", how="left")
sa = sa[sa.assessment_type == "TMA"].sort_values(
    ["id_student","code_module","code_presentation","date"])

def improvement(x):
    return float(x.iloc[-1] - x.iloc[0]) if len(x) > 1 else 0.0

scores = (sa.groupby(["id_student","code_module","code_presentation"])
            .agg(total_score=("score","mean"),
                 std_module_score=("score","std"),
                 score_improvement=("score", improvement),
                 n_assessments=("score","count"))
            .reset_index())
scores["std_module_score"] = scores["std_module_score"].fillna(0)
print("scores aggregated:", scores.shape)

In [ ]:
# CELL 4 - engagement features from studentVle (checkpointed; the slow step)
CKPT = f"{OUT_DIR}/oulad_engagement.parquet"
if os.path.exists(CKPT):
    eng = pd.read_parquet(CKPT); print("loaded engagement from checkpoint")
else:
    vle = pd.read_csv("/content/oulad/studentVle.csv")
    print("VLE rows:", len(vle))
    eng = (vle.groupby(["id_student","code_module","code_presentation"])
              .agg(n_clicks=("sum_click","sum"),
                   n_sessions=("date","nunique"),
                   engagement_span_days=("date", lambda x: int(x.max()-x.min())))
              .reset_index())
    eng.to_parquet(CKPT); print("engagement aggregated:", eng.shape, "-> checkpointed")
print(eng.shape)

In [ ]:
# CELL 5 - merge, and document the population restriction (Section 4.16)
df = (info[["id_student","code_module","code_presentation","gender","imd_band",
            "highest_education","age_band","num_of_prev_attempts","studied_credits",
            "disability","target"]]
        .merge(scores, on=["id_student","code_module","code_presentation"], how="inner")
        .merge(eng,    on=["id_student","code_module","code_presentation"], how="inner"))
df["imd_band"] = df["imd_band"].fillna(MISSING_BAND).astype(str)

raw_withdrawn      = int((info.final_result == "Withdrawn").sum())
kept_withdrawn     = int(df.target.sum())
excluded_withdrawn = raw_withdrawn - kept_withdrawn
print(f"after merge: {df.shape} | positives {df.target.sum():,} ({df.target.mean()*100:.1f}%)")
print("\nPopulation restriction (both TMA and VLE required):")
print(f"  Withdrawn in raw:  {raw_withdrawn:,}")
print(f"  Retained:          {kept_withdrawn:,}")
print(f"  Excluded:          {excluded_withdrawn:,} ({excluded_withdrawn/raw_withdrawn*100:.1f}%)")

print("\nagainst the manuscript:")
check("4.16  merged n", len(df), 24772, 0, "{:.0f}")
check("4.16  at-risk prevalence", df.target.mean(), 0.168, 5e-4, "{:.3f}")
seal("CELL 5")

In [ ]:
# CELL 6 - percentile transform, encoding, imputation
def to_pct(s):
    a = np.asarray(s, dtype=float); r = rankdata(a, method="average")
    return 100.0 * (r - 1) / (len(r) - 1) if len(r) > 1 else np.zeros_like(a)

for col in ["n_clicks", "n_sessions", "engagement_span_days"]:
    df[col + "_pct"] = to_pct(df[col])
df = df.drop(columns=["n_clicks", "n_sessions", "engagement_span_days"])

# source side, in memory only; the .xlsx is never written or committed
source_df  = pd.read_excel(SOURCE_DATA)
source_pct = source_df.copy()
for raw in ["n_module_views", "n_logins", "engagement_span_days"]:
    if raw in source_pct.columns:
        source_pct[raw + "_pct"] = to_pct(source_pct[raw])
        source_pct = source_pct.drop(columns=[raw])

cat_cols = ["gender","imd_band","highest_education","age_band",
            "disability","code_module","code_presentation"]
X = pd.get_dummies(df.drop(columns=["id_student","target"]), columns=cat_cols)
X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X.columns]

# Section 4.16 states this plainly: the means come from all 24,772 records, so
# the imputation saw the held-out partition. The transfer analyses drop these
# records instead; the models here are fitted on OULAD and learn around them.
nan_before = int(X.isna().sum().sum())
X = X.fillna(X.mean(numeric_only=True))
y = df["target"].values
feature_names = X.columns.tolist()
print(f"encoded features: {X.shape[1]} | NaNs before/after imputation: {nan_before} / {int(X.isna().sum().sum())}")

check("4.16  encoded columns", X.shape[1], 43, 0, "{:.0f}")
check("4.16  imputed cells", nan_before, 132, 0, "{:.0f}")
seal("CELL 6")
df.to_parquet(f"{OUT_DIR}/oulad_merged_pct.parquet")

In [ ]:
# CELL 7 - population-shift diagnostics (Section 4.16)
# Percentile ranks are uniform by construction, so PSI on them is ~0 whatever
# the raw distributions look like. Report PSI on the RAW variables.
def psi(a, b, bins=10):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    cuts = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3: return float("nan")
    pa = np.histogram(a, bins=cuts)[0] / len(a) + 1e-6
    pb = np.histogram(b, bins=cuts)[0] / len(b) + 1e-6
    return float(np.sum((pb - pa) * np.log(pb / pa)))

raw_eng = pd.read_parquet(f"{OUT_DIR}/oulad_engagement.parquet")
diag = pd.DataFrame([
    ("Sample size",        f"{len(source_df):,}", f"{len(df):,}"),
    ("At-risk prevalence", f"{source_df.at_risk.eq(0).mean()*100:.1f}%", f"{df.target.mean()*100:.1f}%"),
    ("Encoded features",   "43", str(X.shape[1])),
    ("PSI raw: engagement volume", "-", f"{psi(source_df.n_module_views, raw_eng.n_clicks):.2f}"),
    ("PSI raw: sessions",          "-", f"{psi(source_df.n_logins, raw_eng.n_sessions):.2f}"),
], columns=["Diagnostic","Source","OULAD"])
print(diag.to_string(index=False))
diag.to_csv(f"{OUT_DIR}/step1_diagnostics.csv", index=False)

In [ ]:
# CELL 8 - grid search independently on OULAD, same protocol as the source
Xtr0, Xte0, ytr0, yte0 = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
sc0 = MinMaxScaler(); Xtr0_s = sc0.fit_transform(Xtr0)
Xb0, yb0 = SMOTE(random_state=42).fit_resample(Xtr0_s, ytr0)

grid = {"max_depth":[4,6,8], "n_estimators":[100,200,300],
        "learning_rate":[0.01,0.1], "subsample":[0.8]}
gs = GridSearchCV(XGBClassifier(eval_metric="auc", random_state=42), grid,
                  cv=StratifiedKFold(5, shuffle=True, random_state=42),
                  scoring="roc_auc", n_jobs=-1).fit(Xb0, yb0)
best = gs.best_params_
print("OULAD grid winner:", best, "| CV AUC", round(gs.best_score_, 4))
print("Source winner:     {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 300, 'subsample': 0.8}")

PAPER_BEST = {"learning_rate":0.1, "max_depth":8, "n_estimators":200, "subsample":0.8}
if best != PAPER_BEST:
    raise AssertionError(f"grid winner changed. Table S9 and run_log_oulad.json "
                         f"record {PAPER_BEST}; this run gives {best}")
print("  [CELL 8] grid winner matches Table S9")
json.dump(best, open(f"{OUT_DIR}/oulad_best_params.json","w"), indent=2)

In [ ]:
# CELL 9 - SHAP-guided pruning, then the CANDIDATE fairness removal
# Attributions are computed on the SCALED training inputs the model was fitted
# on, matching Section 5.5. deployed_cols is NOT decided here: Algorithm 1
# line 12 decides it in CELL 11, from the equal-opportunity gaps.
full0 = XGBClassifier(**best, eval_metric="auc", random_state=42).fit(Xb0, yb0)
dm = xgb.DMatrix(Xtr0_s, feature_names=feature_names)
contribs = full0.get_booster().predict(dm, pred_contribs=True)[:, :-1]
mean_abs = pd.Series(np.abs(contribs).mean(axis=0), index=feature_names).sort_values(ascending=False)

cum = mean_abs.cumsum() / mean_abs.sum()
pruned_cols = list(cum[cum <= 0.95].index)
if len(pruned_cols) < len(cum):
    pruned_cols.append(cum.index[len(pruned_cols)])

IMD_COLS = [c for c in pruned_cols if c.lower().startswith("imd_band")]
candidate_cols = [c for c in pruned_cols if c not in IMD_COLS]
print(f"pruning at 95% cumulative |SHAP|: 43 -> {len(pruned_cols)}")
print(f"IMD encodings surviving the prune: {len(IMD_COLS)}  {IMD_COLS}")
print(f"candidate set after removing them: {len(pruned_cols)} -> {len(candidate_cols)}")
print("\nTop 10 by mean |SHAP|:"); print(mean_abs.head(10).round(3).to_string())

check("5.12  pruned columns", len(pruned_cols), 24, 0, "{:.0f}")
check("5.12  candidate columns", len(candidate_cols), 23, 0, "{:.0f}")
if not IMD_COLS:
    _fail.append("no IMD column survived the prune; nothing for the screen to remove")
seal("CELL 9")
mean_abs.to_csv(f"{REPO_OUT}/oulad_shap_values.csv")

In [ ]:
# CELL 10 - shared helpers. Every later cell uses these, so the fitting
# recipe and the threshold rule are defined once.

def fit_o(Xa, ya, cols, seed):
    """Per-split MinMax scaling, then SMOTE, then the OULAD grid winner."""
    sc = MinMaxScaler().fit(Xa[cols])
    Xb, yb = SMOTE(random_state=seed).fit_resample(sc.transform(Xa[cols]), ya)
    return sc, XGBClassifier(**best, eval_metric="auc", random_state=42).fit(Xb, yb)

def val_threshold(Xtr_, ytr_, cols, seed, target=TARGET_RECALL):
    """Highest threshold reaching `target` recall on a 25% VALIDATION split of
    the TRAINING partition (Section 4.4). The test partition is never touched -
    this is the correction that moves tau from 0.4416 to 0.4719."""
    Xi, Xv, yi, yv = train_test_split(Xtr_, ytr_, test_size=0.25,
                                      stratify=ytr_, random_state=seed)
    sc, m = fit_o(Xi, yi, cols, seed)
    pv = m.predict_proba(sc.transform(Xv[cols]))[:, 1]; yv = np.asarray(yv)
    ts = np.sort(np.unique(pv))[::-1]
    rec = np.array([(pv[yv == 1] >= t).mean() for t in ts])
    i = int(np.argmax(rec >= target))
    return float(ts[i]), float(rec[i])

def eligible_bands(imd, y_true):
    """Section 4.9: at least MIN_BAND_N students in the evaluation partition.
    The rule is subgroup SIZE, not the at-risk count. The Missing category is a
    missing-data category rather than a deprivation level, so it is excluded."""
    out = []
    for b in sorted(pd.unique(imd)):
        if b == MISSING_BAND:
            continue
        m = (imd == b)
        if m.sum() >= MIN_BAND_N and (m & (y_true == 1)).sum() > 0:
            out.append(b)
    return out

def band_recall(proba, tau, imd, y_true, bands):
    yh = (np.asarray(proba) >= tau).astype(int)
    return {b: float(yh[(imd == b) & (y_true == 1)].mean()) for b in bands}

def eo_gap(proba, tau, imd, y_true, bands):
    r = band_recall(proba, tau, imd, y_true, bands)
    return max(r.values()) - min(r.values()), r

def mcnemar_correctness(y_true, pa, pb, t=0.5):
    """b = correct by A and wrong by B; c = the reverse. Dietterich (1998), the
    standard McNemar for comparing two classifiers. NOT prediction discordance."""
    yt = np.asarray(y_true)
    A = (np.asarray(pa) >= t).astype(int); B = (np.asarray(pb) >= t).astype(int)
    b_ = int(((A == yt) & (B != yt)).sum()); c_ = int(((A != yt) & (B == yt)).sum())
    x2 = (abs(b_ - c_) - 1) ** 2 / (b_ + c_) if (b_ + c_) else 0.0
    return b_, c_, float(x2), (float(chi2.sf(x2, 1)) if (b_ + c_) else 1.0)

print("helpers defined: fit_o, val_threshold, eligible_bands, band_recall, eo_gap, mcnemar_correctness")

In [ ]:
# CELL 11 - multi-seed AUC, McNemar, and ALGORITHM 1 LINE 12
CONFIGS = {"full": feature_names, "pruned": pruned_cols, "cand": candidate_cols}
rows, seed42 = [], {}
for s in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, stratify=y, random_state=s)
    for name, cols in CONFIGS.items():
        sc, m = fit_o(Xtr, ytr, cols, s)
        p = m.predict_proba(sc.transform(Xte[cols]))[:, 1]
        rows.append(dict(config=name, seed=s, auc=roc_auc_score(yte, p)))
        if s == 42:
            seed42[name] = p
    if s == 42:
        Xtr42, Xte42, ytr42, yte42 = Xtr, Xte, ytr, np.asarray(yte)

per_seed = pd.DataFrame(rows)
print("multi-seed AUC over 5 seeds:")
for cfg, g in per_seed.groupby("config"):
    print(f"  {cfg:<7} {g.auc.mean():.4f} +/- {g.auc.std(ddof=1):.4f}")

b_mc, c_mc, x2_mc, p_mc = mcnemar_correctness(yte42, seed42["full"], seed42["pruned"])
print(f"\n[McNemar, correctness discordance at 0.5] b={b_mc}, c={c_mc}, "
      f"chi2={x2_mc:.3f}, p={p_mc:.3f}")

# --- the thresholds, each from its own validation split -----------------------
tau_pruned, vrec_pruned = val_threshold(Xtr42, ytr42, pruned_cols, 42)
tau_cand,   vrec_cand   = val_threshold(Xtr42, ytr42, candidate_cols, 42)
print(f"\ntau, 24-column pruned:    {tau_pruned:.4f}  (validation recall {vrec_pruned:.3f})")
print(f"tau, 23-column candidate: {tau_cand:.4f}  (validation recall {vrec_cand:.3f})")

imd42 = df.loc[Xte42.index, "imd_band"].astype(str).values
BANDS = eligible_bands(imd42, yte42)
print(f"\nbands audited: {len(BANDS)}  {BANDS}")
print(f"bands excluded: {sorted(set(imd42) - set(BANDS))}")

gap_pruned, rec_pruned = eo_gap(seed42["pruned"], tau_pruned, imd42, yte42, BANDS)
gap_cand,   rec_cand   = eo_gap(seed42["cand"],   tau_cand,   imd42, yte42, BANDS)

# ---- ALGORITHM 1, LINE 12 ----------------------------------------------------
# Accept the removal only if it does not increase the equal-opportunity gap.
accept_removal = bool(gap_cand <= gap_pruned)
deployed_cols  = candidate_cols if accept_removal else pruned_cols
label = (f"{len(deployed_cols)}-column candidate (IMD removed)" if accept_removal
         else f"{len(deployed_cols)}-column pruned (removal REJECTED)")
print(f"\nEO gap {gap_pruned:.4f} -> {gap_cand:.4f}   change {gap_cand-gap_pruned:+.5f}")
print(f">>> Algorithm 1 line 12: accept_removal = {accept_removal}")
print(f">>> OULAD configuration of record: {label}")

print("\nagainst Section 5.12 and Table S9:")
check("5.12  full 5-seed AUC",  per_seed[per_seed.config=="full"].auc.mean(),   0.936, 5e-4, "{:.3f}")
check("5.12  pruned 5-seed AUC", per_seed[per_seed.config=="pruned"].auc.mean(), 0.936, 5e-4, "{:.3f}")
check("5.12  McNemar b", b_mc, 102, 0, "{:.0f}")
check("5.12  McNemar c", c_mc,  97, 0, "{:.0f}")
check("5.12  McNemar p", p_mc, 0.777, 1e-3, "{:.3f}")
check("5.12  tau, 24-column",  tau_pruned, 0.4719, 1e-3)
check("5.12  tau, 23-column",  tau_cand,   0.4874, 1e-3)
check("5.12  bands audited", len(BANDS), 10, 0, "{:.0f}")
check("5.12  EO gap, pruned",    gap_pruned, 0.1185, 1e-3)
check("5.12  EO gap, candidate", gap_cand,   0.1601, 1e-3)
check("5.12  change on removal", gap_cand - gap_pruned, 0.0416, 1e-3)
check("5.12  band recall, lowest",  min(rec_pruned.values()), 0.679, 1e-3, "{:.3f}")
check("5.12  band recall, highest", max(rec_pruned.values()), 0.797, 1e-3, "{:.3f}")
if accept_removal:
    _fail.append("Algorithm 1 line 12 accepted the removal; Section 5.12 reports it rejected")
seal("CELL 11")

In [ ]:
# CELL 12 - operating point, bootstrap interval, and the seed-42 file
yh = (seed42["pruned"] >= tau_pruned).astype(int)
tp = int(((yh == 1) & (yte42 == 1)).sum()); fn = int(((yh == 0) & (yte42 == 1)).sum())
fp = int(((yh == 1) & (yte42 == 0)).sum()); tn = int(((yh == 0) & (yte42 == 0)).sum())
print(f"operating point at tau = {tau_pruned:.4f}: TP {tp}, FN {fn}, FP {fp}, TN {tn}")
print(f"  recall {tp/(tp+fn):.3f}   precision {tp/(tp+fp):.3f}")

# paired percentile bootstrap of the CHANGE in the gap, bands fixed on the full sample
rng = np.random.default_rng(42)
idx = np.arange(len(yte42)); draws = []
for _ in range(1000):
    i = rng.choice(idx, len(idx), replace=True)
    try:
        a, _ = eo_gap(seed42["pruned"][i], tau_pruned, imd42[i], yte42[i], BANDS)
        b, _ = eo_gap(seed42["cand"][i],   tau_cand,   imd42[i], yte42[i], BANDS)
        draws.append(b - a)
    except ValueError:
        pass
ci_lo, ci_hi = float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))
print(f"  change {gap_cand-gap_pruned:+.4f}, 95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]  ({len(draws)} draws)")

print("\nagainst Section 5.12:")
check("5.12  true positives",  tp, 614, 0, "{:.0f}")
check("5.12  false negatives", fn, 217, 0, "{:.0f}")
check("5.12  false positives", fp, 364, 0, "{:.0f}")
check("5.12  true negatives",  tn, 3760, 0, "{:.0f}")
check("5.12  recall at tau",    tp/(tp+fn), 0.739, 1e-3, "{:.3f}")
check("5.12  precision at tau", tp/(tp+fp), 0.628, 1e-3, "{:.3f}")
check("5.12  held-out n", len(yte42), 4955, 0, "{:.0f}")
check("5.12  at-risk n", int(yte42.sum()), 831, 0, "{:.0f}")
check("5.12  bootstrap CI low",  ci_lo, -0.0582, 2e-3)
check("5.12  bootstrap CI high", ci_hi,  0.1233, 2e-3)
seal("CELL 12")

o42 = pd.DataFrame({"y_true": yte42, "proba_full": seed42["full"],
                    "proba_pruned": seed42["pruned"], "proba_cand": seed42["cand"],
                    "imd_band": imd42})
o42.to_csv(f"{REPO_OUT}/oulad_seed42_predictions.csv", index=False)
per_seed.to_csv(f"{REPO_OUT}/oulad_per_seed_metrics.csv", index=False)
print(f"\n[wrote] {REPO_OUT}/oulad_seed42_predictions.csv  ({len(o42)} rows)")
print(f"[wrote] {REPO_OUT}/oulad_per_seed_metrics.csv")

In [ ]:
# CELL 13 - run log, in the schema the acceptance gate reads
run_log = {
    "_note": ("Written by Oulad_exp.ipynb CELL 13. tau is selected on a 25% validation "
              "split of the training partition (Section 4.4), not on the test set. "
              "The configuration of record is the 24-column pruned set: Algorithm 1 "
              "line 12 rejected the removal because the gap rose."),
    "source_file": f"{REPO_OUT}/oulad_seed42_predictions.csv",
    "source_columns": {"pruned_24col": "proba_pruned", "candidate_23col": "proba_cand",
                        "label": "y_true", "protected": "imd_band"},
    "configuration_of_record": "pruned, 24 columns",
    "best_hyperparameters": best,
    "tau_pruned_24col": round(tau_pruned, 4),
    "tau_candidate_23col": round(tau_cand, 4),
    "tau_pruned_24col_full_precision": tau_pruned,
    "tau_candidate_23col_full_precision": tau_cand,
    "val_recall_pruned_24col": vrec_pruned,
    "eo_gap_pruned_24col": round(gap_pruned, 4),
    "eo_gap_candidate_23col": round(gap_cand, 4),
    "eo_change_on_removal": round(gap_cand - gap_pruned, 5),
    "eo_change_ci95": [round(ci_lo, 4), round(ci_hi, 4)],
    "bootstrap": {"draws": 1000, "seed": 42, "scheme": "paired percentile",
                   "bands": (f"{len(BANDS)} recorded deprivation bands, fixed on the full "
                             f"sample. Eligibility is subgroup size, at least {MIN_BAND_N} "
                             f"students in the evaluation partition (Section 4.9). The "
                             f"Missing category is excluded as a missing-data category "
                             f"rather than a deprivation level.")},
    "accept_removal_algorithm1_line12": accept_removal,
    "bands_audited": BANDS,
    "bands_excluded": [MISSING_BAND],
    "missing_band": {"n_records": int((imd42 == MISSING_BAND).sum()),
                      "n_at_risk": int(((imd42 == MISSING_BAND) & (yte42 == 1)).sum()),
                      "excluded": True, "decision": "Section 4.9"},
    "band_recall_range_pruned_24col": [round(min(rec_pruned.values()), 3),
                                        round(max(rec_pruned.values()), 3)],
    "band_recall_pruned_24col": {k: round(v, 3) for k, v in rec_pruned.items()},
    "confusion_at_tau_pruned": {"tp": tp, "fn": fn, "fp": fp, "tn": tn},
    "held_out_n": int(len(yte42)),
    "at_risk_n": int(yte42.sum()),
    "mcnemar_full_vs_pruned": {"b": b_mc, "c": c_mc, "chi2": round(x2_mc, 3),
                                "p": round(p_mc, 3), "discordance": "correctness"},
    "population_restriction": {"withdrawn_raw": raw_withdrawn,
                                "withdrawn_retained": kept_withdrawn,
                                "withdrawn_excluded": excluded_withdrawn},
    "reported_in": "Section 5.12, Table 4, Table S9",
}
json.dump(run_log, open(f"{REPO_OUT}/run_log_oulad.json", "w"), indent=2, default=str)
print(f"[wrote] {REPO_OUT}/run_log_oulad.json")
print(json.dumps({k: run_log[k] for k in
                  ("tau_pruned_24col","tau_candidate_23col","eo_gap_pruned_24col",
                   "eo_gap_candidate_23col","eo_change_on_removal","eo_change_ci95",
                   "accept_removal_algorithm1_line12","bands_excluded")}, indent=2))

In [ ]:
# CELL 14 - ROC figure and the cross-context comparison
plt.rcParams.update({"font.family":"serif","font.size":10,"figure.dpi":300})
fig, ax = plt.subplots(figsize=(5.4, 4.4))
for name, ccol, ls, lab in [("full","#5b7d99","--","Full, 43 columns"),
                            ("pruned","#3d7a4f","-.","Pruned, 24 columns"),
                            ("cand","#1c4a2c","-","Candidate, 23 columns")]:
    fpr, tpr, _ = roc_curve(yte42, seed42[name])
    ax.plot(fpr, tpr, ls, color=ccol, lw=1.8,
            label=f"{lab} (AUC = {roc_auc_score(yte42, seed42[name]):.3f})")
ax.plot([0,1],[0,1],":",color="#999",lw=1,label="Chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(loc="lower right", fontsize=8); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/oulad_FIG_roc.png", dpi=300); plt.show()

# Source figures as PUBLISHED: Table 1 for the 20-split AUCs, Section 5.2 for
# McNemar, Section 5.4 for the equal-opportunity gaps at the screening threshold.
# Both columns are therefore quoted at each configuration's own tau.
comparison = pd.DataFrame({
    "Source": ["3,325", "17.0%", "43 -> 28",
               "0.868 +/- 0.018", "0.871 +/- 0.016", "1.000",
               "0.490 (fee status)", "0.385",
               "module views, logins, score dispersion"],
    "OULAD":  [f"{len(df):,}", f"{df.target.mean()*100:.1f}%",
               f"43 -> {len(pruned_cols)}",
               f"{per_seed[per_seed.config=='full'].auc.mean():.3f} +/- {per_seed[per_seed.config=='full'].auc.std(ddof=1):.3f}",
               f"{per_seed[per_seed.config=='pruned'].auc.mean():.3f} +/- {per_seed[per_seed.config=='pruned'].auc.std(ddof=1):.3f}",
               f"{p_mc:.3f}",
               f"{gap_pruned:.3f} (IMD band)", f"{gap_cand:.3f}",
               ", ".join(mean_abs.head(3).index.tolist())],
}, index=["n", "At-risk prevalence", "Pruning (full -> pruned)",
          "Full AUC (5 seeds)", "Pruned AUC", "McNemar full vs pruned (p)",
          "EO gap, proxy retained", "EO gap, proxy removed", "Top-3 SHAP features"])
print("\n===== CROSS-CONTEXT COMPARISON =====")
print(comparison.to_string())
print("\nNote: on OULAD the removal was REJECTED by Algorithm 1 line 12, so the")
print("configuration of record is the 24-column pruned set. On the source")
print("institution the removal was accepted. Section 5.12 and Section 6.4.")
comparison.to_csv(f"{OUT_DIR}/oulad_vs_source_comparison.csv")

In [ ]:
# CELL 15 - final summary
print("OULAD replication reproduces the manuscript.\n")
print(f"  merged n                 {len(df):,}   prevalence {df.target.mean()*100:.1f}%")
print(f"  columns                  43 -> {len(pruned_cols)} -> {len(candidate_cols)} (candidate)")
print(f"  grid winner              {best}")
print(f"  AUC full / pruned        {per_seed[per_seed.config=='full'].auc.mean():.3f} / "
      f"{per_seed[per_seed.config=='pruned'].auc.mean():.3f}")
print(f"  McNemar                  b={b_mc}, c={c_mc}, p={p_mc:.3f}")
print(f"  tau 24-col / 23-col      {tau_pruned:.4f} / {tau_cand:.4f}")
print(f"  EO gap                   {gap_pruned:.4f} -> {gap_cand:.4f}, change {gap_cand-gap_pruned:+.4f}")
print(f"  95% CI                   [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"  Algorithm 1 line 12      removal {'accepted' if accept_removal else 'REJECTED'}")
print(f"  configuration of record  {label}")
print("\nCommit all four from outputs/: oulad_per_seed_metrics.csv,")
print("oulad_seed42_predictions.csv, run_log_oulad.json, oulad_shap_values.csv.")
print("Never commit model_ready_dataset_1.xlsx, kaggle.json or the Drive parquets.")